In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
from pandas import ExcelWriter
import datetime
from time import sleep
import os

print("Running TH BTHAI Web Scraping Tool v.1.0")
scriptfolder=os.path.dirname(os.path.abspath(__file__))
os.chdir(scriptfolder)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process


now=datetime.datetime.now()

filename= 'TH BTHAI SQL Ready {}.xlsx'.format(str(now).replace(":",".")[:-7])
writer = ExcelWriter(filename)

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
		  'Phone - Mother company': [], 'Check': []}

ints_types=["1. Thai Commercial Banks", "2. Retail Banks", "3. Subsidiary", "4. Foreign Banks Branches", "5. Finance Companies", 
			"6. Credit Fonciers", "7. Foreign Bank Representatives", "8. Assets Management Companies (AMC)", "9. Specialized Financial Institutions"]

driver = webdriver.Chrome()
driver.maximize_window()

driver.get('https://www.bot.or.th/English/FinancialInstitutions/WebsiteFI/Pages/InstList.aspx')

processdate=now.strftime('%Y-%m-%d')

for ints_type in ints_types:
	print(f'Working with TH BTHAI 1 - ({ints_type}).')
	sleep(1)
	for ele in driver.find_elements(By.XPATH, f'//div[@class="table-filter"]'):
		try:
			ele.find_element(By.XPATH, '//select').click()
		except:
			print('not found')
	sleep(1)
	currentdata=driver.page_source
	driver.find_element(By.XPATH, f'//option[@value="{ints_type}"]').click()
	sleep(8)
	while driver.page_source==currentdata and '1' not in ints_type:
		sleep(1)
		print('waiting for page to change')
	#for ele in driver.find_elements(By.XPATH, '//div[@class="gosearch"]'):
	#	try:
	#		ele.click()
	#	except:
	#		print('failed click')
	soup=BeautifulSoup(driver.page_source, 'html.parser')
	div=soup.find_all('div', {'class':'table-responsive'})[-1]
	table=div.find('table', {'class':'table'})
	trs=table.find_all('tr', {'class': 'bg-gray'})
	for tr in trs:
		tds=tr.find_all('td')
		sqldict['ListProcessDate'].append(processdate)
		sqldict['Typology'].append(ints_type.split('.', 1)[1].strip())
		sqldict['Cntry'].append('TH')
		sqldict['Name'].append(tds[1].text.replace(tds[1].find('a').text, '').strip())
		sqldict['Website'].append(tds[1].find('a').text.strip())
		contact=BeautifulSoup(str(tds[2]).replace('<br>', '***').replace('<br/>', '***'), 'html.parser')
		contact=list(filter(lambda x: len(x.strip())>4, contact.text.strip().split('***')))
		address=''
		for cont in contact:
			if 'Tel' not in cont and 'Fax' not in cont:
				address=address+cont
		address=address.replace('BANGKOK', 'Bangkok').replace('Bangkok', ',Bangkok').replace('Bangkok,', 'Bangkok').replace('Bangkok ,', ',Bangkok ')
		address_split=address.split(',')
		address_split=list(filter(lambda x: len(str(x).strip())>0, address_split))
		zipindex=False
		for addix in range(len(address_split)):
			if sum(c.isdigit() for c in address_split[addix])==5:
				zipindex=addix
				break
			else:
				pass
		#print(zipindex, address,  address_split)
		if zipindex:
			sqldict['Address_1'].append(','.join(address_split[:zipindex]).strip())
			sqldict['City'].append(address_split[zipindex].strip().split(' ')[0])
			sqldict['Zip'].append(address_split[zipindex].strip())
		else:
			if 'Thailand' in address:
				address=address.replace('Thailand', '')
			if 'Bangkok' in address:
				address=address.replace('Bangkok', '')
				sqldict['City'].append('Bangkok')
			sqldict['Address_1'].append(address)
		for cont in contact:
			if 'Tel' in cont[:4] and len(cont.replace('Tel.', '').strip())>5:
				sqldict['Phone'].append(cont.replace('Tel.', '').strip())
				break
		for cont in contact:
			if 'Fax' in cont[:4] and len(cont.replace('Fax.', '').strip())>5:
				sqldict['Fax'].append(cont.replace('Fax.', '').strip())
				break
		for key in ['Address_1', 'Phone', 'Fax']:
			if len(sqldict[key])>0:
				if len(sqldict[key][-1])>1:
					while sqldict[key][-1][-1]==',' or sqldict[key][-1][-1]=='.' or sqldict[key][-1][-1]==' ':
						sqldict[key][-1]=sqldict[key][-1][:-1]
		sqldict['RegCtry'].append('TH')
		sqldict['RegCode'].append('THAI')
		sqldict['ListCode'].append('1')
		sqldict['RegulationType'].append('Regulated')
		for key in sqldict.keys():
			while len(sqldict[key])<len(sqldict['ListProcessDate']):
				sqldict[key].append('')


df=pd.DataFrame(sqldict)
df.to_excel(writer, 'SQL ready', index=False)
writer.save()
writer.close()

sleep(3)

driver.quit()
    
    
    

In [ ]:
# check which is different

key_value_counts = {key: len(values) for key, values in sqldict.items()}

# Print the counts
for key, count in key_value_counts.items():
    print(f"Key '{key}' has {count} values.")